In [13]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

text = "Deep learning models are powerful tools for natural language processing and machine translation"
tokens = word_tokenize(text)
print(f"tokens: {tokens}")
without_stopwords = [w for w in tokens if w.lower() not in stopwords.words('english')]
print(f"without_stopwords: {without_stopwords}")
stemmed = [PorterStemmer().stem(w) for w in without_stopwords]
print(f"stemmed: {stemmed}")
lemmatized =[WordNetLemmatizer().lemmatize(w) for w in without_stopwords]
print(f"lemmatized: {lemmatized}")
pos_tagged = pos_tag(tokens)
print(f"pos_tagged: {pos_tagged}")




tokens: ['Deep', 'learning', 'models', 'are', 'powerful', 'tools', 'for', 'natural', 'language', 'processing', 'and', 'machine', 'translation']
without_stopwords: ['Deep', 'learning', 'models', 'powerful', 'tools', 'natural', 'language', 'processing', 'machine', 'translation']
stemmed: ['deep', 'learn', 'model', 'power', 'tool', 'natur', 'languag', 'process', 'machin', 'translat']
lemmatized: ['Deep', 'learning', 'model', 'powerful', 'tool', 'natural', 'language', 'processing', 'machine', 'translation']
pos_tagged: [('Deep', 'NNP'), ('learning', 'NN'), ('models', 'NNS'), ('are', 'VBP'), ('powerful', 'JJ'), ('tools', 'NNS'), ('for', 'IN'), ('natural', 'JJ'), ('language', 'NN'), ('processing', 'NN'), ('and', 'CC'), ('machine', 'NN'), ('translation', 'NN')]


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

corpus=[
    "Deep learning is powerful",
    "Machine translation is powerful",
    "Natural language processing is fun!"
]

#Bag of words
bow=CountVectorizer()
print(f"Bow:\n {bow.fit_transform(corpus).toarray()}")

#TF-IDF
tfidf=TfidfVectorizer()
print(f"tfidf:\n {tfidf.fit_transform(corpus).toarray()}")

Bow:
 [[1 0 1 0 1 0 0 1 0 0]
 [0 0 1 0 0 1 0 1 0 1]
 [0 1 1 1 0 0 1 0 1 0]]
tfidf:
 [[0.5844829  0.         0.34520502 0.         0.5844829  0.
  0.         0.44451431 0.         0.        ]
 [0.         0.         0.34520502 0.         0.         0.5844829
  0.         0.44451431 0.         0.5844829 ]
 [0.         0.47952794 0.28321692 0.47952794 0.         0.
  0.47952794 0.         0.47952794 0.        ]]


In [20]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

sentences =[word_tokenize(doc.lower()) for doc in corpus]
print(f"sentences: {sentences}")

# CBoW: sg=0
chat_model=Word2Vec(sentences, vector_size=50, window=2, min_count=1, sg=0)
print(f"++++++++++++++++++++++++++++")
print(f"chat_model: {chat_model.wv['deep']}")
print(f"++++++++++++++++++++++++++++")
# skip-Gram: sg=1
skip_model=Word2Vec(sentences, vector_size=50, window=2, min_count=1, sg=1)
print(f"skip_model: {skip_model.wv['deep']}")

sentences: [['deep', 'learning', 'is', 'powerful'], ['machine', 'translation', 'is', 'powerful'], ['natural', 'language', 'processing', 'is', 'fun', '!']]
++++++++++++++++++++++++++++
chat_model: [-1.7454965e-02  4.2603230e-03 -1.7470884e-03 -1.8638177e-02
 -1.8856285e-02 -2.8214359e-03  8.8648172e-03  7.4081421e-03
 -1.2997386e-02 -1.3746135e-02 -9.9988244e-03 -4.5736884e-03
 -1.4500575e-02 -1.9206636e-02 -5.4872585e-03 -1.6725682e-02
 -1.2077752e-02 -1.1341858e-02 -4.6882750e-03 -3.4139943e-03
 -1.7913997e-02 -1.4703989e-03  1.6305013e-02  1.5380859e-02
 -1.4412232e-02 -7.3336624e-03  6.2371041e-03 -1.9141445e-02
  2.9528784e-03  1.3048933e-02  1.1492839e-02 -1.7526124e-02
 -9.0342881e-03 -1.6280321e-02  9.1912749e-05  1.8527268e-02
  1.1946611e-02  1.0134616e-02  1.0122125e-02 -6.4858343e-03
  1.9104367e-02 -1.4712849e-02 -1.4540775e-02 -4.5307782e-03
 -1.5571213e-03 -6.4322068e-03 -1.1851717e-03  1.4977646e-02
 -1.3950372e-03 -3.2498813e-03]
++++++++++++++++++++++++++++
skip_model:

In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

# NLP tools
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec

# Deep Learning
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping


nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


np.random.seed(42)
tf.random.set_seed(42)

# ===============================================================
# PART 1: Load Dataset
# ===============================================================
print("Loading IMDB dataset...")

max_features = 10000
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(f"Training data: {len(x_train)} reviews")
print(f"Testing data: {len(x_test)} reviews")

# Get word dictionary to decode reviews
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

# Function to decode review
def decode_review(encoded_review):
    # Skip special tokens and unknown words instead of using '?'
    words = [reverse_word_index.get(i-3, '') for i in encoded_review]
    # Filter out empty strings
    return ' '.join([word for word in words if word])

print("\nSample review excerpt:")
sample_review = decode_review(x_train[0])
print(sample_review[:100] + "...")
print(f"Sentiment: {'Positive' if y_train[0] == 1 else 'Negative'}")

# ===============================================================
# PART 2: Text Preprocessing Demo
# ===============================================================
def preprocess_text(text):
    """Demonstrate NLP preprocessing steps"""
    # Tokenization
    tokens = word_tokenize(text.lower())


    stop_words = set(stopwords.words('english'))
    filtered = [w for w in tokens if w not in stop_words]


    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(w) for w in filtered]


    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(w) for w in filtered]

    return {
        'tokens': tokens[:10],
        'filtered': filtered[:10],
        'stemmed': stemmed[:10],
        'lemmatized': lemmatized[:10]
    }

# Process a sample review
print("\nDemonstrating text preprocessing:")
processed = preprocess_text(sample_review)
for key, value in processed.items():
    print(f"{key.capitalize()}: {value}")

# ===============================================================
# PART 3: Text Vectorization Demo
# ===============================================================
print("\nDemonstrating text vectorization:")

# Create a small corpus for demonstration
small_corpus = [decode_review(x) for x in x_train[:3]]

# Bag of Words
bow = CountVectorizer(max_features=1000)
bow_matrix = bow.fit_transform(small_corpus)
print(f"BoW shape: {bow_matrix.shape}")
print(f"First 5 features: {bow.get_feature_names_out()[:5]}")

# TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf_matrix = tfidf.fit_transform(small_corpus)
print(f"TF-IDF shape: {tfidf_matrix.shape}")

# ===============================================================
# PART 4: Word Embeddings Demo
# ===============================================================
print("\nDemonstrating word embeddings:")

# Tokenize corpus for Word2Vec
tokenized_corpus = [word_tokenize(text.lower()) for text in small_corpus]

# Train Word2Vec models
cbow_model = Word2Vec(tokenized_corpus, vector_size=50, window=2, min_count=1, sg=0)
skipgram_model = Word2Vec(tokenized_corpus, vector_size=50, window=2, min_count=1, sg=1)

# Show embeddings for a sample word
vocab = list(cbow_model.wv.index_to_key)
if vocab:
    sample_word = vocab[0]
    print(f"Embeddings for word '{sample_word}':")
    print(f"CBoW (first 5 dims): {cbow_model.wv[sample_word][:5]}")
    print(f"Skip-gram (first 5 dims): {skipgram_model.wv[sample_word][:5]}")

# ===============================================================
# PART 5: LSTM Model for Sentiment Analysis
# ===============================================================
print("\nPreparing data for LSTM model:")

# Pad sequences to ensure uniform length
maxlen = 200
x_train_pad = pad_sequences(x_train, maxlen=maxlen)
x_test_pad = pad_sequences(x_test, maxlen=maxlen)
print(f"Training data shape after padding: {x_train_pad.shape}")

# Create LSTM model
print("Creating standard LSTM model")
model = Sequential([
    # Embedding layer converts integer indices to dense vectors
    # Add input_shape to ensure model is built immediately
    Embedding(max_features, 128, input_shape=(maxlen,)),

    # LSTM layer processes the sequence
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),

    # Output layer with sigmoid activation for binary classification
    Dense(1, activation='sigmoid')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Print summary with parameter counts
print(f"Model has {model.count_params()} total parameters")
# Skip full summary display
# model.summary()

# Train model
print("\nTraining model...")
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    x_train_pad, y_train,
    batch_size=128,
    epochs=3,  # Small number for demonstration
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate model
print("\nEvaluating model...")
loss, accuracy = model.evaluate(x_test_pad, y_test, verbose=0)
print(f"Test accuracy: {accuracy:.4f}")

# Generate predictions
y_pred = (model.predict(x_test_pad) > 0.5).astype(int).flatten()

# Show classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))



Loading IMDB dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Training data: 25000 reviews
Testing data: 25000 reviews
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Sample review excerpt:
this film was just brilliant casting location scenery story direction everyone's really suited the p...
Sentiment: Positive

Demonstrating text preprocessing:
Tokens: ['this', 'film', 'was', 'just', 'brilliant', 'casting', 'location', 'scenery', 'story', 'direction']
Filtered: ['film', 'brilliant', 'casting', 'location', 'scenery', 'story', 'direction', 'everyone', "'s", 'really']
Stemmed: ['film', 'brilliant', 'cast', 'locat', 'sceneri', 'stori', 'direct', 'everyon', "'s", 'realli']
Lemmatized: ['film', 'brilliant', 'casting', 'location', 'scenery', 'story', 'direction', 'everyone', "'s", 'really']

Demonstrating text vectorization:
BoW shape: (3, 260)
First 5 features: ['1990s' '80' 'abomination' 'across' 'acting']
TF-IDF shape: (3, 260)

Demonstrating word embeddings:
Embeddings fo

/Users/kiara/Documents/Amit-MS-Stuff/NLP_and_GenAI_Course/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:126: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 102s 651ms/step - accuracy: 0.8631 - loss: 0.3346 - val_accuracy: 0.8424 - val_loss: 0.3714
Epoch 3/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 90s 573ms/step - accuracy: 0.8876 - loss: 0.2749 - val_accuracy: 0.8360 - val_loss: 0.3762

Evaluating model...
Test accuracy: 0.8376
782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 49ms/step

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.88      0.84     12500
    Positive       0.87      0.79      0.83     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000

